# Live Data Callbacks

This example demonstrates how to use the [zahner_link](https://doc.zahner.de/im7/apis/zahner_link/python/) library's event-driven callback system to receive **live measurement data** as it is being recorded. In addition to waiting for a job to finish and retrieving all the data at once, callbacks provide real-time access to each data row, header, and status update as they occur.

This is particularly useful for:

* Monitoring long-running measurements - see data as it arrives, not just at the end
* Building custom GUIs - update plots or status indicators in real time
* Logging and diagnostics - track exactly when each measurement phase starts, progresses, and finishes

This notebook covers:
* **Registering event handlers:** Using [ZahnerLinkEventHandler](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkEventHandler) factory methods to create callbacks for DC and EIS live data
* **DC live data callbacks:** Observing header, data rows, and finished events during an [OcvJob](https://doc.zahner.de/im7/apis/zahner_link/python/pages/meas.html#zahner_link.meas.OcvJob) measurement
* **EIS live data callbacks:** Observing EIS-specific header, impedance row, wave, and finished events during an [EisGenerateJob](https://doc.zahner.de/im7/apis/zahner_link/python/pages/meas.html#zahner_link.meas.EisGenerateJob) measurement
* **Job status updates:** Tracking when a job starts, is running, and completes
* **Connection status:** Monitoring the connection state via [set_connection_status_callback()](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc.set_connection_status_callback) (see [ErrorHandling.ipynb](https://doc.zahner.de/im7/apis/zahner_link/python/examples/ErrorHandling.html) for a detailed example of connection error handling)
* **Cleanup:** Properly removing event handlers with [clear_event_handler()](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc.clear_event_handler) before disconnecting

All callback print statements include a **relative timestamp** (seconds since notebook start) so you can see exactly when each event occurs.

In [1]:
import time
import zahner_link as zl
import numpy as np

START_TIME = time.perf_counter()

def elapsed() -> str:
    """Return a formatted string with seconds elapsed since notebook start."""
    return f"{time.perf_counter() - START_TIME:>8.3f}s"

## Defining Callback Functions

Each callback receives specific event data from the library. We define one function per event type:

- `on_connection_status_changed` - called when the connection state transitions (connected, disconnected). Receives a [ZahnerLinkConnectionStatusEnum](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkConnectionStatusEnum)
- `on_live_data_header` - called once at the start of a DC measurement job with a [LiveDataHeader](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.LiveDataHeader) describing the data columns and a [JobInfo](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.JobInfo) identifying the job
- `on_live_data_header_eis` - same concept for EIS measurements, providing a [LiveDataHeaderEis](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.LiveDataHeaderEis) with impedance and path header information
- `on_live_data_rows` - delivers one or more rows of DC live data as a list of lists of floats
- `on_live_data_row_eis` - delivers a single EIS data point as a [LiveDataRowEis](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.LiveDataRowEis) containing impedance values
- `on_live_data_wave` - delivers raw waveform data for EIS as a [LiveDataWavesEis](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.LiveDataWavesEis) object
- `on_live_data_finished` - signals that a job's live data stream has ended
- `on_job_status_update` - reports job lifecycle transitions (queued, running, finished) via [JobStatusEnum](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.JobStatusEnum)

In [2]:
def on_connection_status_changed(status: zl.ZahnerLinkConnectionStatusEnum):
    print(f"[{elapsed()}] [CONNECTION] Status changed: {status}")


def on_live_data_header(header: zl.LiveDataHeader, job_info: zl.JobInfo):
    column_names = [column.get_dimension() for column in header.get_columns()]
    print(
        f"[{elapsed()}] [HEADER DC] Job {job_info.get_job_id()}"
        f" - type: {header.get_short_type()}, columns: {column_names}"
    )


def on_live_data_header_eis(header: zl.LiveDataHeaderEis, job_info: zl.JobInfo):
    impedance_columns = [column.get_dimension() for headers in header.get_impedances_headers() for column in headers.get_columns()]
    meta_columns = [column.get_dimension() for column in header.get_meta_data_header()]
    print(
        f"[{elapsed()}] [HEADER EIS] Job {job_info.get_job_id()}"
        f" - impedance headers length: {len(header.get_impedances_headers())} impedance columns: {impedance_columns} meta columns: {meta_columns}"
    )


def on_live_data_rows(rows: list[list[float]]):
    text = f"[{elapsed()}] [DC ROW]"
    for row in rows:
        text += f" {row}"
    print(text)


def on_live_data_row_eis(row: zl.LiveDataRowEis):
    print(f"[{elapsed()}] [EIS ROW] impedances: {row.get_impedances()} meta: {row.get_meta()}")


def on_live_data_wave(waves: zl.LiveDataWavesEis):
    print(
        f"[{elapsed()}] [EIS WAVE] position={waves.get_position()},"
        f" {len(waves.get_waves())} wave channels"
    )


def on_live_data_finished(job_info: zl.JobInfo):
    print(f"[{elapsed()}] [FINISHED] Job {job_info.get_job_id()} live data complete")


def on_job_status_update(status: zl.JobStatusEnum, job_info: zl.JobInfo):
    print(f"[{elapsed()}] [JOB STATUS] Job {job_info.get_job_id()}: {status}")

## Connecting and Registering Event Handlers

We create a [ZahnerLinkExc](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc) instance and register all event handlers connecting. Each handler is created via a static factory method on [ZahnerLinkEventHandler](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkEventHandler) and then passed to [set_event_handler()](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc.set_event_handler).

The connection status callback is set separately via [set_connection_status_callback()](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc.set_connection_status_callback).

We keep references to all handlers in a list so we can cleanly remove them later.

In [3]:
link = zl.ZahnerLinkExc("10.10.253.150", "1994")

# Register connection status callback
link.set_connection_status_callback(on_connection_status_changed)

# Create and register all event handlers
handlers = [
    zl.ZahnerLinkEventHandler.make_live_data_header_event_handler(on_live_data_header),
    zl.ZahnerLinkEventHandler.make_live_data_header_eis_event_handler(on_live_data_header_eis),
    zl.ZahnerLinkEventHandler.make_live_data_rows_event_handler(on_live_data_rows),
    zl.ZahnerLinkEventHandler.make_live_data_row_eis_event_handler(on_live_data_row_eis),
    zl.ZahnerLinkEventHandler.make_live_data_wave_event_handler(on_live_data_wave),
    zl.ZahnerLinkEventHandler.make_live_data_finished_event_handler(on_live_data_finished),
    zl.ZahnerLinkEventHandler.make_job_status_update_event_handler(on_job_status_update),
]

for handler in handlers:
    link.set_event_handler(handler)

# Connect
print(f"[{elapsed()}] Connecting...")
error: zl.ErrorObject = link.connect()

if not error:
    print(f"[{elapsed()}] Connected successfully")
else:
    print(f"[{elapsed()}] Failed to connect: {error.get_error_code_enum()}")

[   0.021s] Connecting...
[   0.024s] Connected successfully
[   0.024s] [CONNECTION] Status changed: ZahnerLinkConnectionStatusEnum.CONNECTED


## OCV Measurement - Observing DC Live Data

An [OcvJob](https://doc.zahner.de/im7/apis/zahner_link/python/pages/meas.html#zahner_link.meas.OcvJob) measures the Open Circuit Voltage over time. This is a DC measurement, so the following callbacks will fire:

1. **`on_live_data_header`** - once at the start, describing the data columns
2. **`on_live_data_rows`** - repeatedly during the measurement, delivering data rows
3. **`on_live_data_finished`** - once when the measurement is done
4. **`on_job_status_update`** - at each job lifecycle transition

Watch the timestamps to see how the data arrives in real time while [do_job()](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc.do_job) blocks until the job completes.

**Note that live DC data may be incomplete depending on the output data rate.**

In [4]:
ocv_job = zl.meas.OcvJob(duration=5, output_data_rate=5)

print(f"[{elapsed()}] Starting OCV measurement...")
link.do_job(ocv_job)
print(f"[{elapsed()}] OCV measurement complete")

result = link.get_job_result_data(ocv_job)
print(f"[{elapsed()}] OCV result: {result.get_row_count()} data points")

[   0.033s] Starting OCV measurement...
[   0.035s] [JOB STATUS] Job 06c42b5d-1dab-43f9-8a35-46dab1c2f52f: JobStatusEnum.PENDING
[   0.035s] [JOB STATUS] Job 06c42b5d-1dab-43f9-8a35-46dab1c2f52f: JobStatusEnum.RUNNING
[   0.035s] [HEADER DC] Job 06c42b5d-1dab-43f9-8a35-46dab1c2f52f - type: ocv, columns: ['time', 'voltage', 'current', 'shunt']
[   0.240s] [DC ROW] [0.2, -0.00010937730808857705, -6.684560288316892e-11, 11.0]
[   0.640s] [DC ROW] [0.4, -0.00014749760645954284, 5.859794670546467e-13, 11.0]
[   0.846s] [DC ROW] [0.6, -0.00014247600429531535, 4.3931734872539164e-13, 11.0]
[   1.041s] [DC ROW] [0.8, -0.00014879932274871637, 4.912575847217919e-13, 11.0]
[   1.245s] [DC ROW] [1.0, -0.00014670780107060608, 4.563612422716026e-13, 11.0]
[   1.440s] [DC ROW] [1.2, -0.0001448210437525905, 4.335327774508962e-13, 11.0]
[   1.646s] [DC ROW] [1.4, -0.0001440702410989099, 4.2701143844385563e-13, 11.0]
[   1.841s] [DC ROW] [1.6, -0.0001415691907006102, 3.719918911550231e-13, 11.0]
[   2.0

## EIS Measurement - Observing EIS Live Data

Now we run an [EisGenerateJob](https://doc.zahner.de/im7/apis/zahner_link/python/pages/meas.html#zahner_link.meas.EisGenerateJob) in galvanostatic mode. This triggers the EIS-specific callbacks:

1. **`on_live_data_header_eis`** - once at the start, describing impedance headers, path headers, and potentiostat headers
2. **`on_live_data_row_eis`** - for each measured frequency point, delivering impedance data
3. **`on_live_data_wave`** - for each frequency point, delivering the raw waveform data
4. **`on_live_data_finished`** - once when the EIS spectrum is complete
5. **`on_job_status_update`** - at each job lifecycle transition

Before starting the EIS measurement, we switch on the potentiostat in galvanostatic mode using [SwitchOnJob](https://doc.zahner.de/im7/apis/zahner_link/python/pages/control.html#zahner_link.control.SwitchOnJob). Afterwards, we switch it off with [SwitchOffJob](https://doc.zahner.de/im7/apis/zahner_link/python/pages/control.html#zahner_link.control.SwitchOffJob).

In [5]:
main_pot = "MAIN:1:POT"
switch_on_job = zl.control.SwitchOnJob(
    potentiostat=main_pot,
    coupling=zl.PotentiostatCoupling.POTENTIOSTATIC,
    bias=0,
    voltage_range_index=0,
    compliance_range_index=0,
)
switch_off_job = zl.control.SwitchOffJob(potentiostat=main_pot)

print(f"[{elapsed()}] Switching off potentiostat...")
link.do_job(switch_off_job)

print(f"[{elapsed()}] Switching on potentiostat (galvanostatic, 0 A)...")
link.do_job(switch_on_job)

eis_job = zl.meas.EisGenerateJob(
    bias=0.0,
    min_frequency=10,
    max_frequency=10e3,
    start_frequency=1000,
    points_per_decade_upper=10,
    points_per_decade_lower=10,
    pre_duration=0.0,
    pre_waves=1,
    meas_duration=0.1,
    meas_waves=3,
    amplitude=0.01,
)

print(f"[{elapsed()}] Starting EIS measurement...")
link.do_job(eis_job)
print(f"[{elapsed()}] EIS measurement complete")

print(f"[{elapsed()}] Switching off potentiostat...")
link.do_job(switch_off_job)

[   5.046s] Switching off potentiostat...
[   5.091s] Switching on potentiostat (galvanostatic, 0 A)...
[   6.372s] Starting EIS measurement...
[   6.374s] [JOB STATUS] Job cd95f72e-09f4-4ff6-8a99-ab274322a829: JobStatusEnum.PENDING
[   6.375s] [JOB STATUS] Job cd95f72e-09f4-4ff6-8a99-ab274322a829: JobStatusEnum.RUNNING
[   6.707s] [HEADER EIS] Job cd95f72e-09f4-4ff6-8a99-ab274322a829 - impedance headers length: 1 impedance columns: ['|impedance|', 'phase', 'impedance error', 'phase error', 'impedance drift', 'phase drift'] meta columns: ['frequency', 'periods', 'time']
[   7.427s] [EIS ROW] impedances: [[282.65488612860753, -0.4959276986037121, 0.030378253606784256, 9.900392944311998e-05, -0.0041894284908607915, 9.931223285408741e-08]] meta: [1000.0, 3.0, 0.7260000705718994]
[   7.537s] [EIS ROW] impedances: [[262.6359117394668, -0.5643713294628673, 0.01602030438291043, 7.558123261497408e-05, 0.016838091702339852, 0.0001890546041151686]] meta: [1316.8731278794617, 3.0, 0.8369998931884

## Retrieving EIS Result Data

After the measurement completes, we can still retrieve the full result dataset using [get_job_result_data()](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc.get_job_result_data). This returns an [EisDataset](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.EisDataset) from which we extract the frequencies and complex impedances.

The callbacks provided the data in real time. Here we show the final consolidated result, which can then be saved in the Zahner file format.

In [6]:
eis_data = link.get_job_result_data(eis_job)

impedance_data = eis_data.get_impedance_data()
complex_impedances = impedance_data.get_calculated_complex_impedance_track()
frequencies = eis_data.get_frequencies()

print(f"EIS result: {len(frequencies)} frequency points\n")
for freq, z in zip(frequencies, complex_impedances):
    print(f"  {freq:>8.2f} Hz  |Z| = {np.abs(z):>6.2f} Ohm  phase = {np.angle(z,deg=True):>6.2f}°")

EIS result: 39 frequency points

   1000.00 Hz  |Z| = 282.65 Ohm  phase = -28.41°
   1316.87 Hz  |Z| = 262.64 Ohm  phase = -32.34°
   1657.85 Hz  |Z| = 241.82 Ohm  phase = -36.08°
   2087.10 Hz  |Z| = 217.72 Ohm  phase = -39.84°
   2627.51 Hz  |Z| = 191.75 Ohm  phase = -43.21°
   3307.84 Hz  |Z| = 165.75 Ohm  phase = -45.75°
   4164.32 Hz  |Z| = 141.44 Ohm  phase = -47.11°
   5242.57 Hz  |Z| = 120.00 Ohm  phase = -47.14°
   6600.00 Hz  |Z| = 102.06 Ohm  phase = -45.78°
  10000.00 Hz  |Z| =  78.52 Ohm  phase = -40.20°
   6600.00 Hz  |Z| = 102.06 Ohm  phase = -45.79°
   5242.57 Hz  |Z| = 120.01 Ohm  phase = -47.14°
   4164.32 Hz  |Z| = 141.44 Ohm  phase = -47.12°
   3307.84 Hz  |Z| = 165.74 Ohm  phase = -45.74°
   2627.51 Hz  |Z| = 191.76 Ohm  phase = -43.22°
   2087.10 Hz  |Z| = 217.73 Ohm  phase = -39.84°
   1657.85 Hz  |Z| = 241.80 Ohm  phase = -36.07°
   1316.87 Hz  |Z| = 262.67 Ohm  phase = -32.33°
   1046.03 Hz  |Z| = 279.75 Ohm  phase = -29.01°
    830.89 Hz  |Z| = 293.47 Ohm  pha

## Cleanup - Removing Event Handlers and Disconnecting

Before disconnecting, we remove all registered event handlers using [clear_event_handler()](https://doc.zahner.de/im7/apis/zahner_link/python/pages/fundamental.html#zahner_link.ZahnerLinkExc.clear_event_handler) and clear the connection status callback by passing `None`.

After disconnecting, you can no longer execute new jobs, but any data already collected (like the EIS result above) remains valid and can still be used.

In [7]:
for handler in handlers:
    link.clear_event_handler(handler)

link.set_connection_status_callback(None)
link.disconnect()
print(f"[{elapsed()}] Disconnected")

[  15.362s] Disconnected
